## Setup

In [ ]:
from latincyreaders import UDReader, LatinUDReader, PROIELReader
from pprint import pprint
from itertools import islice

In [ ]:
# See all available Latin UD treebanks

treebanks = LatinUDReader.available_treebanks()
for name, description in treebanks.items():
    print(f"  {name:10} - {description}")

In [ ]:
reader = PROIELReader()

## File Discovery

In [ ]:
reader.fileids()[:8]

In [ ]:
len(reader.fileids())

## Metadata

In [ ]:
# Key metadata fields from a sample fileid

fileid = reader.fileids()[0]
doc = next(reader.docs(fileids=fileid))
print(f"fileid:   {doc._.fileid}")
print(f"metadata: {doc._.metadata}")
print(f"tokens:   {len(doc)}")
print(f"sents:    {len(doc.spans.get('ud_sents', []))}")

## Core Interface

### texts()

In [ ]:
next(reader.texts())[:500]

### docs()

In [ ]:
doc = next(reader.docs())
doc

In [ ]:
doc._.fileid, doc._.metadata

### sents()

In [ ]:
sent = next(reader.sents())
sent

In [ ]:
sent._.citation, sent._.metadata

### tokens()

In [ ]:
list(reader.tokens())[:8]

In [ ]:
tok = next(reader.tokens())
tok.text, tok.lemma_, tok.pos_, tok.dep_

## UD Features

### ud_sents()

In [ ]:
for sent in islice(reader.ud_sents(), 5):
    print(f"{sent._.citation}: {sent.text[:70]}...")

### Corpus statistics

In [ ]:
total_sents = 0
total_tokens = 0

for doc in reader.docs():
    total_sents += len(doc.spans.get('ud_sents', []))
    total_tokens += len(doc)

print(f"PROIEL Treebank Statistics:")
print(f"  Files: {len(reader.fileids())}")
print(f"  Sentences: {total_sents:,}")
print(f"  Tokens: {total_tokens:,}")

In [ ]:
from collections import Counter

pos_counts = Counter()
for doc in reader.docs():
    for token in doc:
        pos_counts[token.pos_] += 1

print("POS Tag Distribution (top 5):")
for pos, count in pos_counts.most_common(5):
    print(f"  {pos:<8} {count:>8,}")

### token._.ud — 10 CoNLL-U columns

In [10]:
# Examine token UD annotations

token = doc[0]
print(f"Token: {token.text}")
print()
print("UD annotations (token._.ud):")
pprint(token._.ud)

Token: videns

UD annotations (token._.ud):
{'deprel': 'advcl',
 'deps': None,
 'feats': {'Case': 'Nom',
           'Gender': 'Neut',
           'Number': 'Sing',
           'Tense': 'Pres',
           'VerbForm': 'Part',
           'Voice': 'Act'},
 'form': 'videns',
 'head': 4,
 'id': 1,
 'lemma': 'video',
 'misc': {'Ref': 'MATT_5.1'},
 'upos': 'VERB',
 'xpos': 'V-'}


In [11]:
# Compare UD data with spaCy attributes
# Both are populated from the gold UD annotations

print(f"{'Token':<12} {'lemma_':<12} {'pos_':<8} {'dep_':<10} {'ud[feats]'}")
print("-" * 70)

for token in doc[:10]:
    feats = token._.ud.get('feats', {})
    feats_str = ', '.join(f"{k}={v}" for k, v in feats.items()) if feats else '-'
    print(f"{token.text:<12} {token.lemma_:<12} {token.pos_:<8} {token.dep_:<10} {feats_str}")

Token        lemma_       pos_     dep_       ud[feats]
----------------------------------------------------------------------
videns       video        VERB     advcl      Case=Nom, Gender=Neut, Number=Sing, Tense=Pres, VerbForm=Part, Voice=Act
autem        autem        ADV      discourse  -
turbas       turba        NOUN     obj        Case=Acc, Gender=Fem, Number=Plur
ascendit     ascendo      VERB     root       Mood=Ind, Number=Sing, Person=3, Tense=Pres, VerbForm=Fin, Voice=Act
in           in           ADP      case       -
montem       mons         NOUN     obl        Case=Acc, Gender=Masc, Number=Sing
et           et           CCONJ    cc         -
cum          cum          SCONJ    mark       -
sedisset     sedeo        VERB     advcl      Mood=Sub, Number=Sing, Person=3, Tense=Pqp, VerbForm=Fin, Voice=Act
accesserunt  accedo       VERB     conj       Aspect=Perf, Mood=Ind, Number=Plur, Person=3, Tense=Past, VerbForm=Fin, Voice=Act


In [12]:
# Access morphological features

print("Tokens with Case feature:")
for token in doc[:20]:
    feats = token._.ud.get('feats', {})
    if 'Case' in feats:
        print(f"  {token.text}: {feats['Case']}")

Tokens with Case feature:
  videns: Nom
  turbas: Acc
  montem: Acc
  eum: Acc
  discipuli: Nom
  eius: Gen
  aperiens: Nom
  os: Acc
  suum: Acc
  eos: Acc


### Dependency structure

In [14]:
# Dependency structure is preserved

sent = doc.spans["ud_sents"][0]
print(f"Sentence: {sent.text}")
print()
print(f"{'Token':<12} {'Head':<12} {'Deprel':<10}")
print("-" * 35)
for token in sent:
    print(f"{token.text:<12} {token.head.text:<12} {token.dep_:<10}")

Sentence: videns autem turbas ascendit in montem et cum sedisset accesserunt ad eum discipuli eius et aperiens os suum docebat eos dicens

Token        Head         Deprel    
-----------------------------------
videns       ascendit     advcl     
autem        ascendit     discourse 
turbas       videns       obj       
ascendit     ascendit     root      
in           montem       case      
montem       ascendit     obl       
et           accesserunt  cc        
cum          sedisset     mark      
sedisset     accesserunt  advcl     
accesserunt  ascendit     conj      
ad           eum          case      
eum          accesserunt  obl       
discipuli    accesserunt  nsubj     
eius         discipuli    det       
et           docebat      cc        
aperiens     docebat      advcl     
os           aperiens     obj       
suum         os           det       
docebat      ascendit     conj      
eos          docebat      obj       
dicens       docebat      advcl     


### NER bootstrapping from PROPN tokens

In [ ]:
propn_tokens = [t for t in doc if t.pos_ == "PROPN"]
print(f"Proper nouns in document: {len(propn_tokens)}")
print()
for t in propn_tokens[:5]:
    print(f"  {t.text} (id: {t._.ud.get('id', '?')})")

In [ ]:
# Find sentences containing proper nouns (candidates for annotation)
# PROPN is a heuristic - these need human review!

ner_candidates = []

for sent in reader.ud_sents():
    propns = [t for t in sent if t.pos_ == "PROPN"]
    if propns:
        ner_candidates.append({
            'citation': sent._.citation,
            'text': sent.text,
            'propn_hints': [t.text for t in propns],  # hints, not labels
        })

print(f"Sentences with PROPN tokens (candidates for annotation): {len(ner_candidates)}")

In [ ]:
for item in ner_candidates[:5]:
    print(f"{item['citation']}")
    print(f"  Text: {item['text'][:70]}...")
    print(f"  PROPN hints: {item['propn_hints']}")
    print()

In [ ]:
import json

# Sample export (JSONL for Label Studio, Prodigy, etc.)
for item in ner_candidates[:3]:
    print(json.dumps(item, ensure_ascii=False))

### LatinUDReader — all treebanks at once

In [15]:
# Create a reader for specific treebanks
# (Set auto_download=False to skip download prompts in demo)

# unified = LatinUDReader(treebanks=["proiel", "perseus"])
# for sent in islice(unified.ud_sents(), 10):
#     print(f"{sent._.citation}: {sent.text[:60]}...")

In [16]:
# Download all treebanks at once (run manually when ready)

# LatinUDReader.download_all()